In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import datetime

In [2]:
def sigmam_big(s):
    two_s = int(np.round(2*s))
    prefact = np.sqrt(two_s)
    a_HP = destroy(two_s+1)
    n_op = a_HP.dag() * a_HP
    sqrt_term = (identity(two_s+1) - n_op/(two_s)).sqrtm()
    return prefact*sqrt_term * a_HP

In [3]:
import qutip as qt
from qutip import Qobj, identity, sigmax, sigmay, sigmaz, sigmam, tensor, destroy
from qutip.core.superoperator import liouvillian, sprepost
from qutip.qip.operations import hadamard_transform
#QuTiP control modules
import qutip.control.pulseoptim as cpo

example_name = 'Lindblad'

In [4]:
Nfock = 5
Nspin = 0.5
idOp = tensor(identity(int(2*Nspin)+1), identity(Nfock))
a = tensor(identity(int(2*Nspin)+1), destroy(Nfock))
Sm = tensor(sigmam(), identity(Nfock))
Sx = Sm + Sm.dag()
Sy = 1j*(Sm - Sm.dag())
Sz = Sm.dag()*Sm - Sm*Sm.dag()

H_couple_real = a*Sm.dag() + a.dag()*Sm
H_couple_imag = 1j*(a*Sm.dag() - a.dag()*Sm)

# Uncontrolled Hamiltonian
H0 = 0.0*idOp

#Amplitude damping#
#Damping rate:
gamma = 0.0
L0 = liouvillian(H0, [np.sqrt(gamma)*Sm])

#sigma X control
LC_x = liouvillian(Sx)
#sigma Y control
LC_y = liouvillian(Sy)

LC_couple_real = liouvillian(H_couple_real)
LC_couple_imag = liouvillian(H_couple_imag)


#Drift
drift = L0
#Controls - different combinations can be tried
ctrls = [LC_x, LC_y, LC_couple_real, LC_couple_imag]
# Number of ctrls
n_ctrls = len(ctrls)

# start point for the map evolution
E0 = tensor(qt.basis(2,0), qt.basis(Nfock,0))#sprepost(idOp, idOp)
E0 = E0*E0.dag()

# target for map evolution
E_targ = tensor(qt.basis(2,0), qt.basis(Nfock,3))#sprepost(, )
E_targ = E_targ*E_targ.dag()

# Number of time slots
n_ts = 10
# Time allowed for the evolution
evo_time = 2

# Fidelity error target
fid_err_targ = 1e-3
# Maximum iterations for the optisation algorithm
max_iter = 200
# Maximum (elapsed) time allowed in seconds
max_wall_time = 30
# Minimum gradient (sum of gradients squared)
# as this tends to 0 -> local minima has been found
min_grad = 1e-20

# pulse type alternatives: RND|ZERO|LIN|SINE|SQUARE|SAW|TRIANGLE|
p_type = 'RND'

#Set to None to suppress output files
f_ext = "{}_n_ts{}_ptype{}.txt".format(example_name, n_ts, p_type)
np.Inf = np.inf
result = cpo.optimize_pulse(drift, ctrls, E0, E_targ, n_ts, evo_time, 
                fid_err_targ=fid_err_targ, min_grad=min_grad, 
                max_iter=max_iter, max_wall_time=max_wall_time, 
                out_file_ext=f_ext, init_pulse_type=p_type, gen_stats=True)

ValueError: shapes (100,100) and (10,10) not aligned: 100 (dim 1) != 10 (dim 0)